# Notebook 4 — Modèle BK N blocs : séries temporelles et portraits de phase

Reproduction des Figures 3, 4, 5 d'Erickson et al. (2011) + extensions scientifiques.

**Système :** N blocs couplés avec friction RSF, formulation non-dimensionnelle (éq. 9).  
**Paramètres :** γ_μ = 0.5, γ_λ = √0.2, ξ = 0.5, f̃ = 3.2, ε = 0.5

**Contenu :**
1. **Figures 3–5** : reproduction Erickson (slip law, N=3/9/20)
2. **Extension A — Aging law** : comparaison slip vs aging pour N=20
3. **Extension B — Hétérogénéité spatiale** : aspérité unique (ε variable par bloc)
4. **Extension C — Charge marémotrique** : σ_n(t) sinusoïdal

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import gc

# ══════════════════════════════════════════════════════════════════════════════
#  Dormand–Prince RK45 coefficients
# ══════════════════════════════════════════════════════════════════════════════
A_tab = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [1/5, 0, 0, 0, 0, 0, 0],
    [3/40, 9/40, 0, 0, 0, 0, 0],
    [44/45, -56/15, 32/9, 0, 0, 0, 0],
    [19372/6561, -25360/2187, 64448/6561, -212/729, 0, 0, 0],
    [9017/3168, -355/33, 46732/5247, 49/176, -5103/18656, 0, 0],
    [35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0],
])
B5 = np.array([35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0])
B4 = np.array([5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40])
C  = np.array([0, 1/5, 3/10, 4/5, 8/9, 1, 1])


def rk45_step(f, t, y, h):
    n_stages = 7
    k = np.zeros((n_stages, len(y)))
    k[0] = f(t, y)
    for i in range(1, n_stages):
        yi = y + h * np.dot(A_tab[i, :i], k[:i])
        k[i] = f(t + C[i] * h, yi)
    y5 = y + h * np.dot(B5, k)
    y4 = y + h * np.dot(B4, k)
    err = np.abs(y5 - y4)
    return y5, err, k[0]


def rk45_integrate(f, t_span, y0, rtol=1e-8, atol=1e-10,
                   h_init=1e-3, h_min=1e-14, h_max=10.0,
                   max_steps=100_000_000, store_every=1):
    t0, tf = t_span
    y = np.array(y0, dtype=float)
    t = t0
    h = min(h_init, tf - t0)

    chunk = 200_000
    t_list = np.zeros(chunk)
    y_list = np.zeros((chunk, len(y)))
    t_list[0] = t
    y_list[0] = y
    idx = 1
    n_alloc = chunk
    step_count = 0

    safety = 0.9
    n_steps = 0
    n_reject = 0

    while t < tf:
        if n_steps >= max_steps:
            print(f"  WARNING: hit max_steps={max_steps:,}")
            break

        h = min(h, tf - t)
        if h < h_min:
            h = h_min

        y_new, err, _ = rk45_step(f, t, y, h)

        scale = atol + rtol * np.maximum(np.abs(y), np.abs(y_new))
        err_norm = np.sqrt(np.mean((err / scale)**2))

        if err_norm <= 1.0:
            t = t + h
            y = y_new
            n_steps += 1
            step_count += 1

            if step_count % store_every == 0:
                if idx >= n_alloc:
                    t_list = np.concatenate([t_list, np.zeros(chunk)])
                    y_list = np.concatenate([y_list, np.zeros((chunk, len(y)))])
                    n_alloc += chunk
                t_list[idx] = t
                y_list[idx] = y
                idx += 1

            if err_norm < 1e-30:
                h = h * 5
            else:
                h = h * safety * err_norm**(-0.2)
            h = min(h, h_max)
        else:
            h = h * safety * err_norm**(-0.25)
            h = max(h, h_min)
            n_reject += 1

    t_list = t_list[:idx]
    y_list = y_list[:idx]
    print(f"  {n_steps:,} steps ({n_reject:,} rejected), {idx:,} stored")
    return t_list, y_list


# ══════════════════════════════════════════════════════════════════════════════
#  N-block BK model — non-dimensional eq. (9) from Erickson et al. (2011)
# ══════════════════════════════════════════════════════════════════════════════

def make_nblock_rhs(N, gamma_mu, gamma_lam, xi, eps, f_bar,
                     law='slip', eps_array=None,
                     tidal_delta=0.0, tidal_period=100.0):
    """
    RHS of the BK N-block model (Erickson 2011 eq. 9).

    Additional parameters :
      law         : 'slip' (Ruina 1983, défaut) ou 'aging' (Dieterich 1979)
      eps_array   : array shape (N,) pour spatial heterogeneity de ε.
                    Si None, utilise eps (scalaire) pour tous les blocs.
      tidal_delta : amplitude relative des marées.
                    σ_n(t) = σ_n0·[1 + delta·sin(2π·t/T_tide)]
                    → ξ(t) = ξ0 / (1 + delta·sin(2π·t/T_tide))
      tidal_period: période adimensionnée de la marée.

    Aging law non-dimensionnalisée :
      Θ̄'_j = (1+ε_j)·[exp(-Θ̄_j/(1+ε_j)) - (v̄_j+1)]
    """
    gm2 = gamma_mu**2
    gl2 = gamma_lam**2
    xi0 = xi

    eps_vec  = np.asarray(eps_array, dtype=float) if eps_array is not None else np.full(N, eps, dtype=float)
    eps1_vec = 1.0 + eps_vec

    def rhs(t, y):
        u  = y[:N]
        v  = y[N:2*N]
        Th = y[2*N:3*N]

        dy = np.empty(3*N)
        dy[:N] = v

        lap = np.empty(N)
        lap[0]    = u[1] - u[0]
        lap[1:-1] = u[:-2] - 2*u[1:-1] + u[2:]
        lap[-1]   = u[-2] - u[-1]

        vp1 = np.maximum(v + 1.0, 1e-30)
        lv  = np.log(vp1)

        xi_eff = (xi0 / (1.0 + tidal_delta * np.sin(2*np.pi*t/tidal_period))
                  if tidal_delta != 0.0 else xi0)

        coeff_fric = gm2 / xi_eff
        dy[N:2*N]  = gm2 * lap - gl2 * u - coeff_fric * (f_bar + Th + lv)

        if law == 'slip':
            dy[2*N:3*N] = -vp1 * (Th + eps1_vec * lv)
        else:  # aging (Dieterich 1979)
            # Clip to prevent exp overflow when Θ̄ << 0 during fast slip
            dy[2*N:3*N] = eps1_vec * (np.exp(np.minimum(-Th / eps1_vec, 100.0)) - vp1)

        return dy

    return rhs


def simulate_nblock(N, gamma_mu, gamma_lam, xi, eps, f_bar, duration,
                    rtol=1e-8, atol=1e-10, h_max=1.0, store_every=1,
                    law='slip', eps_array=None,
                    tidal_delta=0.0, tidal_period=100.0):
    rhs = make_nblock_rhs(N, gamma_mu, gamma_lam, xi, eps, f_bar,
                          law=law, eps_array=eps_array,
                          tidal_delta=tidal_delta, tidal_period=tidal_period)

    u_o   = -f_bar * gamma_mu**2 / (xi * gamma_lam**2)
    print(f"  N={N}, u_o = {u_o:.4f}")

    x_bar    = (np.arange(1, N+1) - 0.5) * (20.0 / N)
    u_init   = u_o + np.exp(-(x_bar - 10.0)**2)
    y0       = np.concatenate([u_init, np.zeros(N), np.zeros(N)])
    print(f"  IC: u range [{u_init.min():.4f}, {u_init.max():.4f}]")

    t, y = rk45_integrate(rhs, [0, duration], y0,
                          rtol=rtol, atol=atol,
                          h_init=1e-3, h_max=h_max,
                          store_every=store_every)

    return t, y[:, :N], y[:, N:2*N], y[:, 2*N:3*N], x_bar, u_init




# ══════════════════════════════════════════════════════════════════════════════
#  Parameters from Erickson Figs 3–5
# ══════════════════════════════════════════════════════════════════════════════
gamma_mu  = 0.5
gamma_lam = np.sqrt(0.2)
xi        = 0.5
eps       = 0.5
f_bar     = 3.2


def plot_erickson_figure(N, duration, filename, rtol=1e-8, atol=1e-10,
                         h_max=1.0, store_every=1):
    print(f"\n{'='*60}")
    print(f"  Simulating N={N} blocks, duration={duration}")
    print(f"{'='*60}")

    t, u, v, Theta, x_bar, u_init = simulate_nblock(
        N, gamma_mu, gamma_lam, xi, eps, f_bar, duration,
        rtol=rtol, atol=atol, h_max=h_max, store_every=store_every)

    # Center block index
    mid = (N - 1) // 2

    fig = plt.figure(figsize=(14, 12))

    # ── (a) Initial data ────────────────────────────────────────────
    ax_a = fig.add_subplot(2, 2, 1)
    ax_a.plot(x_bar, u_init, 'bo', markersize=6)
    ax_a.set_xlabel(r'$\bar{x}_j$')
    ax_a.set_ylabel(r'$\bar{u}_j(0)$')
    ax_a.set_title(f'Initial Data for {N} Blockk System')
    ax_a.set_xlim(0, 20)
    ax_a.grid(True, alpha=0.3)

    # ── (b) 3D waterfall of all blocks' slip vs time ────────────────
    ax_b = fig.add_subplot(2, 2, 2, projection='3d')

    max_pts = 3000
    if len(t) > max_pts:
        idx_dec = np.linspace(0, len(t)-1, max_pts, dtype=int)
    else:
        idx_dec = np.arange(len(t))

    t_dec = t[idx_dec]
    u_dec = u[idx_dec, :]

    for j in range(N):
        ax_b.plot(t_dec, np.full_like(t_dec, x_bar[j]), u_dec[:, j],
                  lw=0.4, alpha=0.6)

    ax_b.set_xlabel(r'$\bar{t}$', labelpad=10)
    ax_b.set_ylabel(r'$\bar{x}_j$', labelpad=10)
    ax_b.set_zlabel(r'$\bar{u}_j$', labelpad=5)
    ax_b.set_title(f'Slip for {N} Blockk System')
    ax_b.view_init(elev=25, azim=-50)

    # ── (c) Center block slip vs time (after transient) ──────────────
    ax_c = fig.add_subplot(2, 2, 3)

    # Keep roughly last 70% to see settled behavior
    t_trans = 0.3 * duration
    mask = t > t_trans
    ax_c.plot(t[mask], u[mask, mid], 'b-', lw=0.8)
    ax_c.set_xlabel(r'$\bar{t}$')
    ax_c.set_ylabel(r'$\bar{u}_{%d}(\bar{t})$' % (mid+1))
    ax_c.set_title(f'Contour for Center Blockk of {N} Blockk System')
    ax_c.grid(True, alpha=0.3)

    # ── (d) Phase space of center block ─────────────────────────────
    ax_d = fig.add_subplot(2, 2, 4, projection='3d')

    u_mid = u[mask, mid]
    v_mid = v[mask, mid]
    T_mid = Theta[mask, mid]

    max_phase_pts = 8000
    if len(u_mid) > max_phase_pts:
        idx_ph = np.linspace(0, len(u_mid)-1, max_phase_pts, dtype=int)
    else:
        idx_ph = np.arange(len(u_mid))

    # Phase space: axes order (v, u, Θ) to match Erickson layout
    ax_d.plot(v_mid[idx_ph], u_mid[idx_ph], T_mid[idx_ph], lw=0.3, alpha=0.6)
    ax_d.set_xlabel(r'$\bar{v}_{%d}(\bar{t})$' % (mid+1), labelpad=8)
    ax_d.set_ylabel(r'$\bar{u}_{%d}(\bar{t})$' % (mid+1), labelpad=8)
    ax_d.set_zlabel(r'$\bar{\Theta}_{%d}(\bar{t})$' % (mid+1), labelpad=5)
    ax_d.set_title(f'Phase Space for Center Blockk of {N} Blockk System')
    ax_d.view_init(elev=25, azim=-60)

    fig.suptitle(
        rf'Erickson et al. (2011) — $N={N}$, $\varepsilon={eps}$, '
        rf'$\xi={xi}$, $\gamma_{{\mu}}={gamma_mu}$, '
        rf'$\gamma_{{\lambda}}=\sqrt{{0.2}}$, $\bar{{f}}={f_bar}$',
        fontsize=12, fontweight='bold', y=1.02)

    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved: {filename}")

    del t, u, v, Theta
    gc.collect()


# ══════════════════════════════════════════════════════════════════════════════
#  Run: N=3 (Fig 3), N=9 (Fig 4), N=20 (Fig 5)
# ══════════════════════════════════════════════════════════════════════════════

# Fig 3: N=3, periodic — duration ~300
plot_erickson_figure(N=3, duration=300, filename='fig3_N3.png',
                     rtol=1e-8, atol=1e-10, h_max=0.5)

# Fig 4: N=9, periodic with period doubling — duration ~1500
plot_erickson_figure(N=9, duration=1500, filename='fig4_N9.png',
                     rtol=1e-8, atol=1e-10, h_max=0.5)

# Fig 5: N=20, chaotic — duration ~1500
plot_erickson_figure(N=20, duration=1500, filename='fig5_N20.png',
                     rtol=1e-8, atol=1e-10, h_max=0.5, store_every=1)

print("\nAll done!")


  Simulating N=3 blocks, duration=300
  N=3, u_o = -8.0000
  IC: u range [-8.0000, -7.0000]
  2,485 steps (136 rejected), 2,486 stored
  Saved: fig3_N3.png

  Simulating N=9 blocks, duration=1500
  N=9, u_o = -8.0000
  IC: u range [-8.0000, -7.0000]
  14,343 steps (917 rejected), 14,344 stored
  Saved: fig4_N9.png

  Simulating N=20 blocks, duration=1500
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  16,287 steps (1,172 rejected), 16,288 stored
  Saved: fig5_N20.png

All done!


---
## Extension A — Aging law (Dieterich 1979) vs Slip law (Ruina 1983)

La **slip law** (papier Erickson) : l'état n'évolue qu'avec le glissement.  
La **aging law** : $\dot{\bar{\Theta}}_j = (1+\varepsilon)[e^{-\bar{\Theta}_j/(1+\varepsilon)} - (\dot{\bar{u}}_j+1)]$

Le terme exponentiel permet à θ d'augmenter même à vitesse nulle (*fault healing* intersismique). On compare séries temporelles et portrait de phase pour N=20, ε=0.5.

In [9]:
"""
Slip law vs Aging law comparison — N=20 blocks, ε=0.5 (Erickson parameters Fig. 5).
"""
import gc

N_cmp = 20
T_cmp = 1500

print("=== Aging law vs Slip law : N=20, ε=0.5 ===")
results_law = {}
for law in ['slip', 'aging']:
    print(f"\n  Simulation {law} law...")
    t_, u_, v_, Th_, xb_, _ = simulate_nblock(
        N_cmp, gamma_mu, gamma_lam, xi, eps, f_bar, T_cmp,
        rtol=1e-8, atol=1e-10, h_max=0.5, store_every=1, law=law)
    results_law[law] = (t_, u_, v_, Th_)
    print(f"    {len(t_)} points, v_max = {v_.max():.3f}")

mid = (N_cmp - 1) // 2

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
colors_law = {'slip': '#1D9E75', 'aging': '#D85A30'}
labels_law = {'slip': 'Slip law (Ruina 1983)', 'aging': 'Aging law (Dieterich 1979)'}

for law in ['slip', 'aging']:
    t_, u_, v_, Th_ = results_law[law]
    c   = colors_law[law]
    lbl = labels_law[law]
    mask = t_ > 0.5 * T_cmp

    axes[0, 0].plot(t_[mask], v_[mask, mid], color=c, lw=0.7, label=lbl, alpha=0.85)
    axes[0, 1].plot(t_[mask], Th_[mask, mid], color=c, lw=0.7, label=lbl, alpha=0.85)

    max_pp = 6000
    idx_pp = np.linspace(np.where(mask)[0][0], len(t_)-1, max_pp, dtype=int)
    axes[1, 0].plot(v_[idx_pp, mid], Th_[idx_pp, mid],
                    color=c, lw=0.3, alpha=0.6, label=lbl)

    if law == 'aging':
        max_st = 1500
        idx_st = np.linspace(np.where(mask)[0][0], len(t_)-1, max_st, dtype=int)
        im = axes[1, 1].imshow(
            v_[np.ix_(idx_st, np.arange(N_cmp))].T,
            aspect='auto', origin='lower',
            extent=[t_[idx_st[0]], t_[idx_st[-1]], 0.5, N_cmp+0.5],
            cmap='RdBu_r', vmin=-0.5, vmax=2.0)
        axes[1, 1].set_title('Kymogram v̄_j — Aging law', fontsize=9)
        axes[1, 1].set_xlabel(r'$\bar{t}$')
        axes[1, 1].set_ylabel('Block j')
        plt.colorbar(im, ax=axes[1, 1], label=r'$\bar{v}_j$')

axes[0, 0].set_xlabel(r'$\bar{t}$'); axes[0, 0].set_ylabel(r'$\bar{v}_{centre}$')
axes[0, 0].set_title('Central block velocity'); axes[0, 0].legend(fontsize=8)
axes[0, 0].grid(True, ls=':', alpha=0.4)

axes[0, 1].set_xlabel(r'$\bar{t}$'); axes[0, 1].set_ylabel(r'$\bar{\Theta}_{centre}$')
axes[0, 1].set_title("State variable central block"); axes[0, 1].legend(fontsize=8)
axes[0, 1].grid(True, ls=':', alpha=0.4)

axes[1, 0].set_xlabel(r'$\bar{v}$'); axes[1, 0].set_ylabel(r'$\bar{\Theta}$')
axes[1, 0].set_title('Phase portrait central block'); axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, ls=':', alpha=0.4)

fig.suptitle(
    r'Slip law vs Aging law — N=20, $\varepsilon=0.5$, $\gamma_\mu=0.5$, $\xi=0.5$',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('nb4_aging_vs_slip_N20.png', dpi=140, bbox_inches='tight')
plt.close(fig)
print("\nFigure → nb4_aging_vs_slip_N20.png")
del results_law; gc.collect()

=== Aging law vs Slip law : N=20, ε=0.5 ===

  Simulation slip law...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  16,287 steps (1,172 rejected), 16,288 stored
    16288 points, v_max = 3.143

  Simulation aging law...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  41,843 steps (1,845 rejected), 41,844 stored
    41844 points, v_max = 5.266

Figure → nb4_aging_vs_slip_N20.png


45764

---
## Extension B — Hétérogénéité spatiale : aspérité unique

Un seul bloc "aspérité" (j_star = N//2) avec ε_asp ≠ ε_bg.  
- ε_asp > ε_bg : aspérité velocity-weakening renforcée → nucléation localisée
- ε_asp < 0 : barrière velocity-strengthening → ruptures partielles

On visualise via un **kymogramme** (v̄_j en fonction du temps et du bloc).

In [7]:
"""
Spatial heterogeneity: single asperity at block j_star = N//2.
Sweep over ε_asp, tous les autres blocs à ε_bg = 0.5.
"""
import gc

N_asp   = 20
T_asp   = 800
EPS_BG  = 0.5
j_star  = N_asp // 2

EPS_ASP_LIST = [-0.3, 0.0, 0.5, 1.5, 3.0]

fig, axes = plt.subplots(len(EPS_ASP_LIST), 1,
                         figsize=(13, 4*len(EPS_ASP_LIST)))
fig.suptitle(
    rf'Single asperity (bloc {j_star+1}/{N_asp}) — $\varepsilon_{{bg}}={EPS_BG}$, N={N_asp}',
    fontsize=12, fontweight='bold')

for row, eps_asp in enumerate(EPS_ASP_LIST):
    eps_arr = np.full(N_asp, EPS_BG)
    eps_arr[j_star] = eps_asp

    regime = 'VS' if eps_asp < 0 else ('homogeneous' if eps_asp == EPS_BG else 'VW↑')
    label  = f'ε_asp = {eps_asp:.1f}  ({regime})'
    print(f'\n  {label}...')

    t_, u_, v_, Th_, xb_, _ = simulate_nblock(
        N_asp, gamma_mu, gamma_lam, xi, EPS_BG, f_bar, T_asp,
        rtol=1e-8, atol=1e-10, h_max=0.5, store_every=1,
        law='slip', eps_array=eps_arr)

    mask   = t_ > 0.3 * T_asp
    idx_st = np.linspace(np.where(mask)[0][0], len(t_)-1, 1200, dtype=int)

    im = axes[row].imshow(
        v_[np.ix_(idx_st, np.arange(N_asp))].T,
        aspect='auto', origin='lower',
        extent=[t_[idx_st[0]], t_[idx_st[-1]], 0.5, N_asp+0.5],
        cmap='RdBu_r', vmin=-0.3, vmax=2.0)
    axes[row].axhline(j_star + 1, color='yellow', lw=1.5, ls='--', label='Asperity')
    axes[row].set_ylabel('Block j', fontsize=9)
    axes[row].set_title(label, fontsize=9, loc='left')
    axes[row].legend(fontsize=7, loc='upper right')
    plt.colorbar(im, ax=axes[row], label=r'$\bar{v}_j$')
    del t_, u_, v_, Th_; gc.collect()

axes[-1].set_xlabel(r'$\bar{t}$', fontsize=10)
plt.tight_layout()
plt.savefig('nb4_heterogeneity_asperity.png', dpi=140, bbox_inches='tight')
plt.close(fig)
print('\nFigure → nb4_heterogeneity_asperity.png')


  ε_asp = -0.3  (VS)...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  9,517 steps (825 rejected), 9,518 stored

  ε_asp = 0.0  (VW↑)...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  9,459 steps (778 rejected), 9,460 stored

  ε_asp = 0.5  (homogène)...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  8,627 steps (637 rejected), 8,628 stored

  ε_asp = 1.5  (VW↑)...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  11,291 steps (1,018 rejected), 11,292 stored

  ε_asp = 3.0  (VW↑)...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  23,222 steps (2,312 rejected), 23,223 stored

Figure → nb4_heterogeneity_asperity.png


---
## Extension C — Charge marémotrique (variation de σ_n)

Variation sinusoïdale de la contrainte normale :
$$\sigma_n(\bar{t}) = \sigma_{n0}\left[1 + \delta\sin\!\left(\frac{2\pi\bar{t}}{T_{tide}}\right)\right]$$

En non-dimensionnel : $\xi(\bar{t}) = \xi_0 / [1 + \delta\sin(2\pi\bar{t}/T_{tide})]$ dans le terme de friction.

**Paramètre clé :** rapport $R = T_{tide}/T_{rec}$ (résonance si R ~ 1).  
On mesure la **phase de déclenchement** : les séismes se déclenchent-ils préférentiellement quand σ_n est max ou min ?

In [8]:
"""
Tidal loading: ξ(t) = ξ0 / (1 + δ·sin(2πt/T_tide)).
Sweep over amplitude δ. Histogramme de phase de déclenchement.
"""
import gc

N_tide    = 20
T_sim     = 1200
T_TIDE    = 50.0   # période adimensionnée (T_rec ≈ 20–40 → R ≈ 1.5)
DELTAS    = [0.0, 0.05, 0.15, 0.30]

def detect_events(t, v, threshold=0.5):
    """Event onset times (v_max crosses threshold upward)."""
    v_max = np.max(np.abs(v), axis=1)
    above = v_max > threshold
    return np.array([t[i] for i in range(1, len(t)-1)
                     if above[i] and not above[i-1]])

fig, axes = plt.subplots(len(DELTAS), 2, figsize=(14, 4.5*len(DELTAS)))
fig.suptitle(
    rf'Tidal loading — N={N_tide}, $T_{{tide}}={T_TIDE}$, slip law',
    fontsize=12, fontweight='bold')

for row, delta in enumerate(DELTAS):
    print(f'\n  δ = {delta:.2f}...')
    t_, u_, v_, Th_, xb_, _ = simulate_nblock(
        N_tide, gamma_mu, gamma_lam, xi, eps, f_bar, T_sim,
        rtol=1e-8, atol=1e-10, h_max=0.5, store_every=1,
        law='slip', tidal_delta=delta, tidal_period=T_TIDE)

    mask   = t_ > 0.3 * T_sim
    idx_st = np.linspace(np.where(mask)[0][0], len(t_)-1, 1500, dtype=int)

    # Kymogram
    im = axes[row, 0].imshow(
        v_[np.ix_(idx_st, np.arange(N_tide))].T,
        aspect='auto', origin='lower',
        extent=[t_[idx_st[0]], t_[idx_st[-1]], 0.5, N_tide+0.5],
        cmap='RdBu_r', vmin=-0.3, vmax=2.0)
    axes[row, 0].set_ylabel('Block j')
    axes[row, 0].set_title(f'δ = {delta:.2f} — kymogram', fontsize=9, loc='left')
    plt.colorbar(im, ax=axes[row, 0], label=r'$\bar{v}_j$')

    # Histogramme de phase
    t_ev = detect_events(t_[mask], v_[mask, :], threshold=0.5)
    if len(t_ev) > 0:
        phases = (t_ev % T_TIDE) / T_TIDE * 360
        axes[row, 1].hist(phases, bins=24, range=(0, 360),
                          color='#378ADD', edgecolor='white', alpha=0.85)
        axes[row, 1].axvline(90,  color='red',  ls='--', lw=1.2, label='σ_n max (90°)')
        axes[row, 1].axvline(270, color='blue', ls='--', lw=1.2, label='σ_n min (270°)')
        axes[row, 1].set_xlabel('Tidal phase (°)')
        axes[row, 1].set_ylabel('Event count')
        axes[row, 1].set_title(
            f'δ = {delta:.2f} — {len(t_ev)} events', fontsize=9, loc='left')
        axes[row, 1].legend(fontsize=7)
        axes[row, 1].grid(True, ls=':', alpha=0.4)
    else:
        axes[row, 1].text(0.5, 0.5, 'No event detected',
                          ha='center', va='center', transform=axes[row, 1].transAxes)

    del t_, u_, v_, Th_; gc.collect()

axes[-1, 0].set_xlabel(r'$\bar{t}$')
plt.tight_layout()
plt.savefig('nb4_tidal_loading.png', dpi=140, bbox_inches='tight')
plt.close(fig)
print('\nFigure → nb4_tidal_loading.png')


  δ = 0.00...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  12,982 steps (941 rejected), 12,983 stored

  δ = 0.05...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  12,868 steps (898 rejected), 12,869 stored

  δ = 0.15...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  12,939 steps (857 rejected), 12,940 stored

  δ = 0.30...
  N=20, u_o = -8.0000
  IC: u range [-8.0000, -7.2212]
  12,904 steps (776 rejected), 12,905 stored

Figure → nb4_tidal_loading.png
